In [1]:
DEVICE = "cpu"
MODEL_PATH = "checkpoints/tuning_2.pth"

In [2]:
import torchaudio

torchaudio.__version__

'2.2.2+cpu'

In [3]:
import torch
import torchaudio
import torch.nn.functional as F
import os
import torch.nn as nn



class SimpleAudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.spec_layer = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_fft=1024, hop_length=256, n_mels=64
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), 
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.spec_layer(x)
        x = self.to_db(x)
        x = self.conv_layers(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

In [4]:
import torch
import torchaudio
import torch.nn.functional as F
import os
from collections import Counter

def predict_audio_voted(file_path):
    print(f"File: {os.path.basename(file_path)}")
    
    # 1. Load & Basic Preprocessing
    waveform, sample_rate = torchaudio.load(file_path)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(sample_rate, 16000)
        waveform = resampler(waveform)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # 2. Segmentasi (Pecah per 4 detik)
    segment_len = 16000 * 4
    total_len = waveform.shape[1]
    
    segments = []
    # Jika audio lebih pendek dari 4 detik, beri padding
    if total_len <= segment_len:
        pad_amount = segment_len - total_len
        segments.append(F.pad(waveform, (0, pad_amount)))
    else:
        # Pecah menjadi segmen-segmen non-overlapping
        for i in range(0, total_len - segment_len + 1, segment_len):
            segments.append(waveform[:, i : i + segment_len])
        
        # Ambil sisa audio di ujung (jika ada) dengan teknik tail-crop
        if total_len % segment_len != 0:
            segments.append(waveform[:, -segment_len:])

    print(f"   => Audio dipecah menjadi {len(segments)} segmen analisis.")

    # 3. Predict Masing-masing Segmen
    model = SimpleAudioCNN().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    segment_preds = []
    with torch.no_grad():
        for i, seg in enumerate(segments):
            seg_input = seg.unsqueeze(0).to(DEVICE)
            output = model(seg_input)
            prob = F.softmax(output, dim=1)
            pred = torch.max(output, 1)[1].item()
            segment_preds.append(pred)
            
            label_seg = "PALSU" if pred == 1 else "ASLI"
            print(f"      Segmen {i+1}: {label_seg} (Confidence: {prob[0][pred]*100:.2f}%)")

    # 4. Majority Voting
    vote_result = Counter(segment_preds)
    final_pred = vote_result.most_common(1)[0][0]
    
    # 5. Output Akhir
    print("-" * 45)
    if final_pred == 1:
        print(f"KESIMPULAN AKHIR: PALSU (DEEPFAKE)")
        print(f"(Berdasarkan voting {vote_result[1]} segmen Palsu vs {vote_result[0]} segmen Asli)")
    else:
        print(f"KESIMPULAN AKHIR: ASLI (REAL)")
        print(f"(Berdasarkan voting {vote_result[0]} segmen Asli vs {vote_result[1]} segmen Palsu)")
    print("-" * 45)


In [5]:
if __name__ == "__main__":
    predict_audio_voted("dataset/private_dataset/REAL/audio20.wav")

File: audio20.wav
   => Audio dipecah menjadi 4 segmen analisis.
      Segmen 1: ASLI (Confidence: 74.73%)
      Segmen 2: ASLI (Confidence: 88.94%)
      Segmen 3: ASLI (Confidence: 85.42%)
      Segmen 4: ASLI (Confidence: 56.08%)
---------------------------------------------
KESIMPULAN AKHIR: ASLI (REAL)
(Berdasarkan voting 4 segmen Asli vs 0 segmen Palsu)
---------------------------------------------


In [6]:
if __name__ == "__main__":
    predict_audio_voted("dataset/tuning/AI/OpenAI/alloy_28.wav")

File: alloy_28.wav
   => Audio dipecah menjadi 1 segmen analisis.
      Segmen 1: PALSU (Confidence: 100.00%)
---------------------------------------------
KESIMPULAN AKHIR: PALSU (DEEPFAKE)
(Berdasarkan voting 1 segmen Palsu vs 0 segmen Asli)
---------------------------------------------


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchaudio
from  sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import random

In [8]:
class DeepfakeDataset(Dataset):
    def __init__(self, list_file, max_duration=4, target_sample_rate=16000, is_train=True):
        self.data = []
        self.max_duration = max_duration
        self.target_sample_rate = target_sample_rate
        self.num_samples = max_duration * target_sample_rate
        self.is_train = is_train 

        if not os.path.exists(list_file):
            raise FileNotFoundError(f" Error: File {list_file} tidak ditemukan!")

        with open(list_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 2:
                    # Gabungkan kembali path jika ada spasi (case jarang, tapi aman)
                    path = " ".join(parts[:-1])
                    label = int(parts[-1])
                    self.data.append((path, label))
    
    def __len__(self):
        return len(self.data)

    def _pad_or_trim(self, waveform):
        channels, length = waveform.shape
        if length > self.num_samples:
            if self.is_train:
                start = torch.randint(0, length - self.num_samples, (1,)).item()
            else:
                start = (length - self.num_samples) // 2
            waveform = waveform[:, start : start + self.num_samples]
        elif length < self.num_samples:
            pad_amount = self.num_samples - length
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))
        return waveform

    def __getitem__(self, idx):
        audio_path, label = self.data[idx]
        try:
            waveform, sample_rate = torchaudio.load(audio_path)
        except Exception as e:
            return torch.zeros(1, self.num_samples), torch.tensor(label, dtype=torch.long)

        if sample_rate != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(sample_rate, self.target_sample_rate)
            waveform = resampler(waveform)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Augmentasi Noise (Penting buat Fine-Tuning)
        if self.is_train and random.random() < 0.5: 
            noise_amp = 0.005 * torch.rand(1).item() * torch.max(waveform)
            waveform = waveform + noise_amp * torch.randn_like(waveform)

        waveform = self._pad_or_trim(waveform)
        return waveform, torch.tensor(label, dtype=torch.long)

In [9]:
MODEL_PATH = "checkpoints/tuning_2_1.pth" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16

BENCHMARKS = [
    ("1. XTTS Baseline (Lab)", "dataset/libri+gen/test_list.txt"),
    ("2. Kaggle Multi-Attack", "dataset/tuning/finetune_test_ultimate.txt"),
    ("3. In-the-Wild (Real World)", "dataset/final_mix_test.txt")
]

def run_full_audit():

    model = SimpleAudioCNN().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()


    all_results = {}

    for name, list_file in BENCHMARKS:
        print(f"\nMenguji: {name}")
        
        if not os.path.exists(list_file):
            print(f"    Skip: File '{list_file}' tidak ditemukan.")
            continue

        # Load Dataset (is_train=False agar hasil konsisten/center crop)
        ds = DeepfakeDataset(list_file, is_train=False)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
        
        y_true = []
        y_pred = []
        
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predicted.cpu().numpy())
        
        acc = accuracy_score(y_true, y_pred) * 100
        all_results[name] = acc
        
        print(f"   Akurasi: {acc:.4f}%")
        print(classification_report(y_true, y_pred, target_names=['Asli', 'Palsu'], digits=4))

    for name, score in all_results.items():
        status = "SANGAT BAIK" if score > 85 else ("CUKUP" if score > 70 else "LEMAH")
        print(f"{name:<30} | {score:.2f}% | {status}")
    print("-" * 65)

if __name__ == "__main__":
    run_full_audit()


Menguji: 1. XTTS Baseline (Lab)
   Akurasi: 95.0000%
              precision    recall  f1-score   support

        Asli     0.9615    0.9434    0.9524        53
       Palsu     0.9375    0.9574    0.9474        47

    accuracy                         0.9500       100
   macro avg     0.9495    0.9504    0.9499       100
weighted avg     0.9502    0.9500    0.9500       100


Menguji: 2. Kaggle Multi-Attack
   Akurasi: 91.0112%
              precision    recall  f1-score   support

        Asli     0.8593    0.9892    0.9197       463
       Palsu     0.9860    0.8244    0.8980       427

    accuracy                         0.9101       890
   macro avg     0.9226    0.9068    0.9088       890
weighted avg     0.9201    0.9101    0.9093       890


Menguji: 3. In-the-Wild (Real World)
   Akurasi: 87.2089%
              precision    recall  f1-score   support

        Asli     0.9400    0.8517    0.8937      4011
       Palsu     0.7814    0.9070    0.8396      2345

    accuracy   